In [1]:
from google.colab import drive
drive.mount('/content/drive')



Mounted at /content/drive


In [2]:
from openpyxl import load_workbook

order_folder = '/content/drive/MyDrive/Samurai/Final_Case_Study/samples/order_new'
import os

file_paths = []

for file in os.listdir(order_folder):
    if file.endswith('.xlsx'):
        path = os.path.join(order_folder, file)
        file_paths.append(path)

print(file_paths)

['/content/drive/MyDrive/Samurai/Final_Case_Study/samples/order_new/order_C_20230524.xlsx', '/content/drive/MyDrive/Samurai/Final_Case_Study/samples/order_new/order_A_20230524.xlsx', '/content/drive/MyDrive/Samurai/Final_Case_Study/samples/order_new/order_D_20230524.xlsx', '/content/drive/MyDrive/Samurai/Final_Case_Study/samples/order_new/order_B_20230524.xlsx']


# ① 注文集計

In [3]:
import pandas as pd

def summarize_order (file_paths):
  orders = pd. DataFrame()

  for file_path in file_paths:
    order = pd.read_excel(file_path)
    orders = pd.concat([orders,order],ignore_index=True)

  total_orders = orders.select_dtypes(include="number").sum()

  return total_orders.to_dict()

print(file_paths)
print(summarize_order(file_paths))


['/content/drive/MyDrive/Samurai/Final_Case_Study/samples/order_new/order_C_20230524.xlsx', '/content/drive/MyDrive/Samurai/Final_Case_Study/samples/order_new/order_A_20230524.xlsx', '/content/drive/MyDrive/Samurai/Final_Case_Study/samples/order_new/order_D_20230524.xlsx', '/content/drive/MyDrive/Samurai/Final_Case_Study/samples/order_new/order_B_20230524.xlsx']
{'トマト': 31.0, 'レタス': 42.0, '白菜': 25.0, '大根': 15.0, 'ニンジン': 32.0, 'キャベツ': 21.0, 'ほうれん草': 23.0}


# ② 在庫取得（最終行）

In [4]:
inventory_file = '/content/drive/MyDrive/Samurai/Final_Case_Study/samples/inventory.xlsx'
pickup_file = '/content/drive/MyDrive/Samurai/Final_Case_Study/samples/pickup.xlsx'

In [5]:
def load_latest_inventory(inventory_file):
    df = pd.read_excel(inventory_file)

    latest = df.iloc[-1]

    inventory = {}

    for col in df.columns:
        if col not in ["日付", "曜日"]:
            val = latest[col]

            if pd.isna(val):
                val = 0

            inventory[col] = int(val)

    return inventory, df

inventory, df = load_latest_inventory(inventory_file)

print(inventory)

{'トマト': 91, 'キャベツ': 73, 'レタス': 103, '白菜': 84, 'ほうれん草': 75, '大根': 48, 'ニンジン': 50}


# ③ 発注判定（pickup横持ち）

In [7]:
def check_reorder(inventory, orders, pickup_file):
    pickup_df = pd.read_excel(pickup_file, index_col=0)

    threshold_row = pickup_df.loc["しきい値"]
    order_amount_row = pickup_df.loc["追加量"]

    reorder_list = {}
    new_inventory = {}

    for item, stock in inventory.items():
        order_qty = orders.get(item, 0)
        remaining = stock - order_qty

        new_inventory[item] = remaining

        threshold = threshold_row.get(item, 0)
        order_amount = order_amount_row.get(item, 0)

        if pd.isna(threshold):
            threshold = 0
        if pd.isna(order_amount):
            order_amount = 0

        if remaining < threshold:
            reorder_list[item] = int(order_amount)

    return reorder_list, new_inventory



④ メール本文

In [8]:
def create_email_body(reorder_list):
    if not reorder_list:
        return "本日は発注不要です。"

    body = "以下の商品を発注してください。\n\n"

    for item, qty in reorder_list.items():
        body += f"{item}：{qty}個\n"

    return body

⑤ メール送信

In [12]:
def send_email(body):
    smtp_host = "sandbox.smtp.mailtrap.io"
    smtp_port = 587
    username = "3a0323097413bf"
    password = "881bfef3661659"

    msg = MIMEText(body)
    msg["Subject"] = "野菜発注依頼"
    msg["From"] = "from@example.com"
    msg["To"] = "to@example.com"

    with smtplib.SMTP(smtp_host, smtp_port) as server:
        server.starttls()
        server.login(username, password)
        server.send_message(msg)

⑥ 在庫更新

In [16]:
def update_inventory(df, new_inventory, file_path):
    from datetime import datetime
    now = datetime.now()

    new_row = {
        "日付": now.strftime("%Y-%m-%d %H:%M:%S"),
        "曜日": now.strftime("%a")
    }

    for item, qty in new_inventory.items():
        new_row[item] = qty

    # 列順を揃える
    new_df = pd.DataFrame([new_row])[df.columns]

    df = pd.concat([df, new_df], ignore_index=True)
    df.to_excel(file_path, index=False)

# メイン処理

In [17]:
def main():
    base_path = "/content/drive/MyDrive/Samurai/Final_Case_Study/samples"

    order_folder = os.path.join(base_path, "order_new")
    inventory_file = os.path.join(base_path, "inventory.xlsx")
    pickup_file = os.path.join(base_path, "pickup.xlsx")

    # ファイル一覧取得
    file_paths = []
    for file in os.listdir(order_folder):
        if file.endswith(".xlsx"):
            file_paths.append(os.path.join(order_folder, file))

    # ① 注文集計
    orders = summarize_order(file_paths)
    print("注文合計:", orders)

    # ② 在庫取得
    inventory, inventory_df = load_latest_inventory(inventory_file)
    print("現在在庫:", inventory)

    # ③ 発注判定
    reorder_list, new_inventory = check_reorder(
        inventory, orders, pickup_file
    )
    print("発注対象:", reorder_list)

    # ④ メール本文
    body = create_email_body(reorder_list)
    print("\n---メール本文---\n")
    print(body)

    # ⑤ メール送信
    # send_email(body)  ←最初はコメント推奨

    # ⑥ 在庫更新
    update_inventory(inventory_df, new_inventory, inventory_file)

    print("\n処理完了！")


# =====================
# 実行
# =====================
if __name__ == "__main__":
    main()

注文合計: {'トマト': 31.0, 'レタス': 42.0, '白菜': 25.0, '大根': 15.0, 'ニンジン': 32.0, 'キャベツ': 21.0, 'ほうれん草': 23.0}
現在在庫: {'トマト': 91, 'キャベツ': 73, 'レタス': 103, '白菜': 84, 'ほうれん草': 75, '大根': 48, 'ニンジン': 50}
発注対象: {'ニンジン': 80}

---メール本文---

以下の商品を発注してください。

ニンジン：80個


処理完了！
